# Layer 3 — Deep Dive

Directed analysis for a specific finding selected from the Layer 2
findings summary.

**Copy this template** for each deep-dive. Update the finding details
and analysis type below.

**Available analysis types:**
- Regression (does A predict B after controlling for confounders?)
- Lag analysis (does X precede Y by N years?)
- Segmented comparison (high-X vs low-X regions on outcome)
- Before/after analysis (metrics before vs after an event year)
- Narrative pattern test (masking, displacement, divergence, etc.)

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pipeline.config import load_config, get_data_processed_dir

PROJECT_NAME = "qol-immigration"
config = load_config(PROJECT_NAME)
processed_dir = get_data_processed_dir(config)

# Load the three analysis panels (same as 01_descriptive / 02_findings)
PANELS = ["state_panel", "national_panel", "district_crosssection"]
datasets = {}
for name in PANELS:
    path = processed_dir / f"{name}.parquet"
    if path.exists():
        datasets[name] = pd.read_parquet(path)

# National-level annual series from state_panel for lag analysis (one row per year)
if "state_panel" in datasets:
    national_agg = (
        datasets["state_panel"]
        .groupby("year")[["foreign_born_pct", "median_household_income", "real_gdp_millions"]]
        .mean()
        .reset_index()
    )
    datasets["state_panel_national"] = national_agg

print(f"Loaded {len(datasets)} dataset(s): {list(datasets.keys())}")

Loaded 4 dataset(s): ['state_panel', 'national_panel', 'district_crosssection', 'state_panel_national']


## Deep-dive 1: QoL vs GDP divergence

Does median household income diverge from GDP growth over time? (Hypothesis: states with higher immigration see greater divergence.)

In [2]:
from pipeline.patterns import run_pattern

results = run_pattern(
    datasets,
    pattern="divergence",
    outcome_var="median_household_income",
    predictor_var="real_gdp_millions",
    time_col="year",
)
print(results["summary"])
for fig in results.get("figures", []):
    fig.show()

Normalized to base year = 100.
Maximum divergence at 2023: gap = 27.8 index points.


## Deep-dive 2: Immigration → income lag

Does foreign-born share lead or lag median household income? (National annual series.)

In [3]:
from pipeline.patterns import run_lag_analysis

results = run_lag_analysis(
    datasets,
    predictor="foreign_born_pct",
    outcome="median_household_income",
    time_col="year",
    max_lag=10,
)
print(results["summary"])
if results.get("figure"):
    results["figure"].show()

Best correlation at lag=0: r=0.4787
Positive lag means foreign_born_pct leads median_household_income by that many years.


## Deep-dive 3: Housing threshold

Is there a tipping point of foreign-born share above which home prices (HPI) jump?

In [4]:
from pipeline.patterns import run_pattern

results = run_pattern(
    datasets,
    pattern="threshold",
    outcome_var="hpi_annual_avg",
    predictor_var="foreign_born_pct",
)
print(results["summary"])
for fig in results.get("figures", []):
    fig.show()

Binned analysis (10 bins):
   x_mean     y_mean  count
 2.176196 293.985788     92
 3.530870 336.818043     92
 4.298043 341.780109     92
 5.150659 337.122857     91
 6.260645 354.357500     93
 7.847912 382.966016     91
 9.955714 424.134066     91
13.240217 476.356168     92
15.934348 529.068940     92
21.993152 514.577174     92

Possible threshold near foreign_born_pct = 5.15: relationship direction changes.


## (Optional) More deep-dives

Template sections below: set ANALYSIS_TYPE and variables to run regression, segmented comparison, before/after, or other patterns.

In [5]:
# Run run_segmented_comparison(datasets, predictor, outcome, segment_var=...) when needed
print("Skipped (optional).")

Skipped (optional).


## Before/After (optional)

In [6]:
# Run run_before_after(datasets, outcome, event_year=...) when needed
print("Skipped (optional).")

Skipped (optional).


## Other patterns (optional)

In [7]:
# Run run_pattern(datasets, pattern="masking"|"displacement"|..., ...) when needed
print("Skipped (optional).")

Skipped (optional).


## Interpretation

*Use a deep-thinking LLM (Claude thinking, o3, Gemini 2.5 Pro) in Cursor
to interpret these results. Paste the key outputs above into the prompt
along with the project hypothesis.*

**LLM interpretation notes:**

(Paste LLM interpretation here after review)

**Human assessment:**

- Defensible? 
- Caveats: 
- Include in story? 